In [ ]:
import customtkinter as ctk
from treys import Card, Evaluator, Deck

# =========================
# CONFIGURAÇÃO
# =========================

ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("blue")

evaluator = Evaluator()

# =========================
# ESTADO
# =========================

cartas_selecionadas = []
botoes_cartas = {}

# =========================
# SÍMBOLOS
# =========================

simbolos = {
    "e": "♠",
    "c": "♥",
    "o": "♦",
    "p": "♣"
}

# =========================
# FUNÇÕES AUXILIARES
# =========================

def formatar_carta_visual(carta):

    valor = carta[0]
    naipe = carta[1]

    simbolo = simbolos[naipe]

    return f"{valor}{simbolo}"

# =========================
# FUNÇÕES POKER
# =========================

def criar_carta(texto):

    texto = texto.strip()

    texto = texto.replace("10", "T")

    valor = texto[0].upper()
    naipe = texto[1].lower()

    conversao = {
        "e": "s",
        "c": "h",
        "o": "d",
        "p": "c"
    }

    naipe = conversao.get(naipe, naipe)

    carta_final = valor + naipe

    return Card.new(carta_final)


def calcular_vitoria(player, board, simulacoes=5000):

    vitorias = 0
    empates = 0

    for _ in range(simulacoes):

        deck = Deck()

        usadas = player + board

        for carta in usadas:
            deck.cards.remove(carta)

        vilao = deck.draw(2)

        mesa_completa = board.copy()

        while len(mesa_completa) < 5:
            mesa_completa.append(deck.draw(1)[0])

        score_player = evaluator.evaluate(mesa_completa, player)
        score_vilao = evaluator.evaluate(mesa_completa, vilao)

        if score_player < score_vilao:
            vitorias += 1

        elif score_player == score_vilao:
            empates += 1

    winrate = (vitorias / simulacoes) * 100
    empate = (empates / simulacoes) * 100

    return winrate, empate


def sugestao_acao(winrate, board):

    if len(board) == 0:

        if winrate < 45:
            return "FOLD"

        elif winrate < 60:
            return "CALL"

        elif winrate < 75:
            return "RAISE"

        else:
            return "RAISE FORTE"

    else:

        if winrate < 35:
            return "FOLD"

        elif winrate < 55:
            return "CALL / CHECK"

        elif winrate < 70:
            return "BET"

        else:
            return "RAISE FORTE"

# =========================
# FUNÇÕES UI
# =========================

def atualizar_interface():

    mao = cartas_selecionadas[:2]
    board = cartas_selecionadas[2:]

    # converte para símbolos visuais
    mao_visual = [formatar_carta_visual(c) for c in mao]
    board_visual = [formatar_carta_visual(c) for c in board]

    mao_label.configure(
        text=
        "Sua mão: " + " ".join(mao_visual) +
        "        |        " +
        "Mesa: " + " ".join(board_visual)
    )

    if len(mao) < 2:

        resultado_label.configure(text="")
        info_extra_label.configure(text="")

        return

    player = [criar_carta(c) for c in mao]
    mesa = [criar_carta(c) for c in board]

    winrate, empate = calcular_vitoria(player, mesa)

    resultado_label.configure(
        text=f"Vitória - {winrate:.2f}%  |  Empate - {empate:.2f}%"
    )

    # =========================
    # TIPO DA MÃO
    # =========================

    mao_atual_texto = ""

    if len(mesa) >= 3:

        rank = evaluator.get_rank_class(
            evaluator.evaluate(mesa, player)
        )

        descricao_en = evaluator.class_to_string(rank)

        traducao_maos = {
            "High Card": "Carta Alta",
            "Pair": "Par",
            "Two Pair": "Dois Pares",
            "Three of a Kind": "Trinca",
            "Straight": "Sequência",
            "Flush": "Flush",
            "Full House": "Full House",
            "Four of a Kind": "Quadra",
            "Straight Flush": "Straight Flush",
            "Royal Flush": "Royal Flush"
        }

        descricao = traducao_maos.get(
            descricao_en,
            descricao_en
        )

        mao_atual_texto = f"Mão atual: {descricao}"

    # =========================
    # SUGESTÃO
    # =========================

    sugestao = sugestao_acao(winrate, mesa)

    info_extra_label.configure(
        text=
        f"{mao_atual_texto}        |        Sugestão: {sugestao}"
    )


def selecionar_carta(carta):

    if len(cartas_selecionadas) >= 7:
        return

    if carta in cartas_selecionadas:
        return

    cartas_selecionadas.append(carta)

    botoes_cartas[carta].configure(
        border_width=4,
        border_color="#3498db"
    )

    atualizar_interface()


# =========================
# RESET
# =========================

def resetar():

    global cartas_selecionadas

    cartas_selecionadas = []

    mao_label.configure(
        text="Sua mão:        |        Mesa:"
    )

    resultado_label.configure(text="")
    info_extra_label.configure(text="")

    for carta, botao in botoes_cartas.items():

        botao.configure(
            border_width=2,
            border_color="#cccccc"
        )

# =========================
# APP
# =========================

app = ctk.CTk()

app.geometry("1450x820")
app.title("Poker Odds Calculator")

# =========================
# BOTÃO NOVA RODADA
# =========================

reset_btn = ctk.CTkButton(
    app,
    text="🔄 NOVA RODADA",
    width=180,
    height=50,
    font=("Arial", 20),
    command=resetar,
    fg_color="#444444",
    hover_color="#666666"
)

reset_btn.place(
    relx=0.97,
    rely=0.03,
    anchor="ne"
)

# =========================
# TÍTULO
# =========================

titulo = ctk.CTkLabel(
    app,
    text="CALCULADORA DE POKER",
    font=("Arial", 36)
)

titulo.pack(pady=10)

# =========================
# INFO
# =========================

mao_label = ctk.CTkLabel(
    app,
    text="Sua mão:        |        Mesa:",
    font=("Arial", 26)
)

mao_label.pack(pady=5)

resultado_label = ctk.CTkLabel(
    app,
    text="",
    font=("Arial", 24)
)

resultado_label.pack(pady=8)

# =========================
# LINHA HORIZONTAL
# =========================

info_extra_label = ctk.CTkLabel(
    app,
    text="",
    font=("Arial", 28)
)

info_extra_label.pack(pady=8)

# =========================
# GRID CARTAS
# =========================
frame_cartas = ctk.CTkFrame(app, fg_color="transparent")

frame_cartas.pack(
    pady=8,
    fill="y",
    expand=True
)

valores = [
    "A", "K", "Q", "J",
    "T", "9", "8", "7",
    "6", "5", "4", "3", "2"
]

naipes_info = [
    ("e", "ESPADAS"),
    ("c", "COPAS"),
    ("o", "OUROS"),
    ("p", "PAUS")
]

for linha, (naipe, nome_naipe) in enumerate(naipes_info):

    label_naipe = ctk.CTkLabel(
        frame_cartas,
        text=nome_naipe,
        font=("Arial", 16)
    )

    label_naipe.grid(
        row=linha * 2,
        column=0,
        columnspan=13,
        pady=(6, 2)
    )

    for coluna, valor in enumerate(valores):

        carta = valor + naipe

        if naipe in ["c", "o"]:
            cor_texto = "#ff2222"
        else:
            cor_texto = "#000000"

        simbolo = simbolos[naipe]

        texto_carta = f"{valor}\n{simbolo}"

        btn = ctk.CTkButton(
            frame_cartas,
            text=texto_carta,

            width=54,
            height=76,

            corner_radius=10,

            font=("Arial", 18, "bold"),

            fg_color="#f2f2f2",
            hover_color="#d9d9d9",

            text_color=cor_texto,

            border_width=2,
            border_color="#cccccc",

            command=lambda c=carta: selecionar_carta(c)
        )

        btn.grid(
            row=(linha * 2) + 1,
            column=coluna,
            padx=2,
            pady=2
        )

        botoes_cartas[carta] = btn

# =========================
# LOOP
# =========================

app.mainloop()